In [ ]:

# IMPORTS


# Standard Library


import os
import re
import json
import pickle
import sqlite3
import hashlib
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict


# Type Hinting


from typing import Any, Dict, List, Tuple, Optional

# Numerical Computing


import numpy as np

# Data Processing


import pandas as pd
from collections import defaultdict

# Natural Language Processing


import spacy

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Sentence Embeddings


from sentence_transformers import SentenceTransformer

# Vector Database (Semantic Search)


import faiss


# Knowledge Graph


import networkx as nx


# NVIDIA API Client


from openai import OpenAI


# Progress Bars

from tqdm import tqdm

print("imports ok")

# ============================================================
# CONFIGURATION
# ============================================================

# Project

PROJECT_NAME = "Personal Memory AI"
PROJECT_VERSION = "1.0.0"

# Directories

NOTES_FOLDER = "notes"
DATABASE_FILE = "personal_memory.db"

# NVIDIA API

NVIDIA_API_KEY = "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"

NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"

LLM_MODEL = "meta/llama-3.3-70b-instruct"

LLM_TEMPERATURE = 0.2
LLM_MAX_TOKENS = 4096

# Embedding Model

EMBEDDING_MODEL = "all-MiniLM-L6-v2"

EMBEDDING_DIMENSION = 384

# spaCy

SPACY_MODEL = "en_core_web_sm"

# TF-IDF

MAX_FEATURES = 1000

NGRAM_RANGE = (1, 2)

STOP_WORDS = "english"

# Importance Scoring

SUMMARY_LENGTH = 3

MIN_NOTE_LENGTH = 30

# Semantic Search

TOP_K_RESULTS = 10

SIMILARITY_THRESHOLD = 0.60

# Memory Consolidation

CONSOLIDATE_EVERY = 25

# Random Seed

RANDOM_SEED = 42

# DATABASE

def initialize_database():

    # Connect to SQLite database

    connection = sqlite3.connect(DATABASE_FILE)

    cursor = connection.cursor()

    # Enable foreign keys

    cursor.execute("PRAGMA foreign_keys = ON")

    # Create Notes table

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS notes (

        note_id INTEGER PRIMARY KEY AUTOINCREMENT,

        note_hash TEXT UNIQUE,

        note_date TEXT,

        original_text TEXT,

        clean_text TEXT,

        summary TEXT,

        importance REAL,

        created_at TEXT

    )
    """)

    # Create Memories table

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS memories (

        memory_id INTEGER PRIMARY KEY AUTOINCREMENT,

        note_id INTEGER,

        memory_type TEXT,

        memory_value TEXT,

        confidence REAL,

        FOREIGN KEY(note_id) REFERENCES notes(note_id)

    )
    """)

    # Create Topics table

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS topics (

        topic_id INTEGER PRIMARY KEY AUTOINCREMENT,

        note_id INTEGER,

        topic TEXT,

        score REAL,

        FOREIGN KEY(note_id) REFERENCES notes(note_id)

    )
    """)

    # Create Entities table

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS entities (

        entity_id INTEGER PRIMARY KEY AUTOINCREMENT,

        note_id INTEGER,

        entity TEXT,

        entity_type TEXT,

        FOREIGN KEY(note_id) REFERENCES notes(note_id)

    )
    """)

    # Create Relationships table

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS relationships (

        relationship_id INTEGER PRIMARY KEY AUTOINCREMENT,

        note_id INTEGER,

        source TEXT,

        relation TEXT,

        target TEXT,

        FOREIGN KEY(note_id) REFERENCES notes(note_id)

    )
    """)

    # Create Embeddings table

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS embeddings (

        note_id INTEGER PRIMARY KEY,

        embedding BLOB,

        FOREIGN KEY(note_id) REFERENCES notes(note_id)

    )
    """)

    # Create Observations table

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS observations (

        observation_id INTEGER PRIMARY KEY AUTOINCREMENT,

        note_id INTEGER,

        observation TEXT,

        FOREIGN KEY(note_id) REFERENCES notes(note_id)

    )
    """)

    # Create Predictions table

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS predictions (

        prediction_id INTEGER PRIMARY KEY AUTOINCREMENT,

        prediction TEXT,

        confidence REAL,

        created_at TEXT

    )
    """)

    # Create Indexes

    cursor.execute("CREATE INDEX IF NOT EXISTS idx_notes_date ON notes(note_date)")

    cursor.execute("CREATE INDEX IF NOT EXISTS idx_memories_type ON memories(memory_type)")

    cursor.execute("CREATE INDEX IF NOT EXISTS idx_topics_topic ON topics(topic)")

    cursor.execute("CREATE INDEX IF NOT EXISTS idx_entities_name ON entities(entity)")

    # Save changes

    connection.commit()

    print("Database initialized successfully.")

    return connection



# MEMORY SYSTEM INITIALIZATION


def initialize_memory_system():

    global nlp

    global embedding_model

    global vector_index

    global knowledge_graph

    global tfidf_vectorizer

    nlp = spacy.load(SPACY_MODEL)

    embedding_model = SentenceTransformer(EMBEDDING_MODEL)

    vector_index = faiss.IndexFlatL2(EMBEDDING_DIMENSION)

    knowledge_graph = nx.MultiDiGraph()

    tfidf_vectorizer = TfidfVectorizer(

        max_features=MAX_FEATURES,

        ngram_range=NGRAM_RANGE,

        stop_words=STOP_WORDS

    )

    print("Memory system initialized.")


# LLM CLIENT

client = OpenAI(

    base_url=NVIDIA_BASE_URL,

    api_key=NVIDIA_API_KEY

)

def call_llm(

    system_prompt,

    user_prompt,

    temperature = LLM_TEMPERATURE,

    max_tokens = LLM_MAX_TOKENS

):

    try:

        response = client.chat.completions.create(

            model = LLM_MODEL,

            temperature = temperature,

            max_tokens = max_tokens,

            messages = [

                {

                    "role": "system",

                    "content": system_prompt

                },

                {

                    "role": "user",

                    "content": user_prompt

                }

            ]

        )

        return response.choices[0].message.content.strip()

    except Exception as error:

        print(f"LLM Error : {error}")

        return None

# NOTES PROCESSING


def import_notes():

    notes = []

    NOTES_FOLDER = "notes"

    notes_path = Path(NOTES_FOLDER)

    if not notes_path.exists():

        raise FileNotFoundError(f"{NOTES_FOLDER} folder not found.")

    note_files = sorted(notes_path.glob("notes_*.txt"))

    print(f"Found {len(note_files)} notes.")

    for file in tqdm(note_files):

        text = file.read_text(

            encoding = "utf-8"

        ).strip()

        notes.append(

            {

                "filename": file.name,

                "text": text

            }

        )

    return notes

# PROCESS NOTES

import re
import json
import hashlib
from datetime import datetime

MIN_NOTE_LENGTH = 10
SUMMARY_LENGTH = 3


def process_note(note):

    start_time = datetime.now()
    raw_text = note.get("text", "")
    filename = note.get("filename", "unknown.txt")

    text = raw_text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[^\x09\x0A\x0D\x20-\x7E\u00A0-\uFFFF]", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = "\n".join(line.strip() for line in text.split("\n"))
    clean = text.strip()

    if len(clean) < MIN_NOTE_LENGTH:
        print(f"Skipping '{filename}' — too short ({len(clean)} chars).")
        return None

    normalized = clean.lower()
    normalized = re.sub(r"[#*_`>~]", "", normalized)
    normalized = re.sub(r"http\S+|www\.\S+", " ", normalized)
    normalized = re.sub(r"[^a-z0-9.,!?'\s]", " ", normalized)
    normalized = re.sub(r"\s{2,}", " ", normalized).strip()

    note_hash = hashlib.sha256(normalized.encode("utf-8")).hexdigest()

    date_match = re.search(r"(\d{4}-\d{2}-\d{2})", filename)
    if not date_match:
        date_match = re.search(r"(\d{4}-\d{2}-\d{2})", clean[:50])
    note_date = date_match.group(1) if date_match else datetime.now().strftime("%Y-%m-%d")

    doc = nlp(clean)
    sentences = list(doc.sents)

    summary_prompt = (
        "You are a precise summarization engine for a personal memory system. "
        f"Read the note and reply with JSON only, no extra text, in this exact shape: "
        f'{{"summary": "at most {SUMMARY_LENGTH} short factual sentences", '
        '"keywords": ["3 to 6 important keywords or short phrases"]}'
    )
    llm_reply = call_llm(summary_prompt, clean)

    summary = ""
    keywords = []
    if llm_reply:
        try:
            parsed = json.loads(llm_reply)
            summary = parsed.get("summary", "")
            keywords = parsed.get("keywords", [])
        except Exception:
            summary = llm_reply

    if not summary:
        fallback_sentences = [s.text.strip() for s in sentences if s.text.strip()]
        summary = " ".join(fallback_sentences[:SUMMARY_LENGTH]) if fallback_sentences else clean[:200]

    length_score = min(len(clean) / 1000, 1.0)
    entity_score = min(len(doc.ents) / 10, 1.0)

    important_words = [
        "decided", "decision", "goal", "deadline", "important",
        "milestone", "promise", "plan", "realized", "learned"
    ]
    combined_text = (clean + " " + summary).lower()
    keyword_hits = sum(1 for word in important_words if word in combined_text)
    keyword_score = min(keyword_hits / 5, 1.0)

    heuristic_score = (0.4 * length_score) + (0.3 * entity_score) + (0.3 * keyword_score)

    importance_prompt = (
        "Rate how important this personal note is for someone's long-term memory, "
        "on a scale of 1 (trivial) to 10 (life-changing). Consider goals, decisions, "
        "relationships, health, career and emotional weight. Respond with ONLY the integer."
    )
    llm_score_reply = call_llm(importance_prompt, f"Note: {clean}\nSummary: {summary}", max_tokens=10)

    llm_score = None
    if llm_score_reply:
        digits = re.findall(r"\d+", llm_score_reply)
        if digits:
            llm_score = max(1, min(10, int(digits[0]))) / 10

    if llm_score is not None:
        importance = round((0.5 * heuristic_score) + (0.5 * llm_score), 4)
    else:
        importance = round(heuristic_score, 4)

    embedding = embedding_model.encode(
        clean, convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")

    word_count = len(clean.split())
    paragraph_count = len([p for p in clean.split("\n\n") if p.strip()])

    if len(clean) < 300:
        length_category = "short"
    elif len(clean) < 1500:
        length_category = "medium"
    else:
        length_category = "long"

    metadata = {
        "filename": filename,
        "note_date": note_date,
        "char_count": len(clean),
        "word_count": word_count,
        "sentence_count": len(sentences),
        "paragraph_count": paragraph_count,
        "entity_count": len(doc.ents),
        "reading_time_minutes": round(word_count / 200, 2),
        "length_category": length_category,
        "is_short_note": len(clean) < MIN_NOTE_LENGTH,
        "processed_at": datetime.now().isoformat(),
        "processing_seconds": round((datetime.now() - start_time).total_seconds(), 4)
    }

    return {
        "filename": filename,
        "note_hash": note_hash,
        "note_date": note_date,
        "original_text": raw_text,
        "clean_text": clean,
        "normalized_text": normalized,
        "summary": summary,
        "keywords": keywords,
        "importance": importance,
        "embedding": embedding,
        "spacy_doc": doc,
        "metadata": metadata
    }

# MEMORY EXTRACTION


import json
import re
import uuid
from datetime import datetime, timezone
from typing import Any, Dict, List

import spacy

nlp = spacy.load("en_core_web_sm")

# Fields the LLM returns that are always simple lists of short strings.
MEMORY_FIELDS = (
    "goals",
    "projects",
    "beliefs",
    "skills",
    "interests",
    "events",
    "tasks",
    "emotions",
    "preferences",
)

def run_structural_pass(text: str) -> Dict[str, Any]:
    """Use spaCy to find entities and simple subject-verb-object relations."""
    doc = nlp(text)

    entities = [{"text": ent.text, "label": ent.label_} for ent in doc.ents]

    relations = []
    for token in doc:
        if token.dep_ in ("nsubj", "nsubjpass") and token.head.pos_ in ("VERB", "AUX"):
            obj = next(
                (child.text for child in token.head.children
                 if child.dep_ in ("dobj", "attr", "prep", "obj", "acomp", "xcomp")),
                None,
            )
            relations.append({"subject": token.text, "predicate": token.head.lemma_, "object": obj})

    noun_chunks = list(dict.fromkeys(chunk.text for chunk in doc.noun_chunks))

    return {"entities": entities, "relations": relations, "noun_chunks": noun_chunks}


def build_prompt(text: str, scaffold: Dict[str, Any]) -> str:
    """Build the LLM prompt, grounded with the spaCy scaffold."""
    return f"""You are the Memory Extraction engine of a Personal Memory AI System.
Read the note below and convert it into a structured JSON memory object.
Use the linguistic scaffold as grounding evidence — prefer it over inventing
new entities — but you may add items that are clearly implied even if not
explicitly tagged.

NOTE TEXT:
\"\"\"{text}\"\"\"

DETECTED ENTITIES (spaCy NER):
{json.dumps(scaffold["entities"], ensure_ascii=False)}

DETECTED SUBJECT-VERB-OBJECT RELATIONS (dependency parsing):
{json.dumps(scaffold["relations"], ensure_ascii=False)}

DETECTED NOUN PHRASES:
{json.dumps(scaffold["noun_chunks"], ensure_ascii=False)}

Return ONLY a single valid JSON object (no markdown fences, no commentary)
with exactly these keys:

{{
  "topics": [string],
  "goals": [string],
  "projects": [string],
  "beliefs": [string],
  "skills": [string],
  "interests": [string],
  "observations": [string],
  "events": [string],
  "tasks": [string],
  "entities": {{
      "people": [string],
      "organizations": [string],
      "locations": [string],
      "dates": [string],
      "other": [string]
  }},
  "relationships": [{{"subject": string, "predicate": string, "object": string}}],
  "emotions": [string],
  "preferences": [string],
  "importance_score": float between 0 and 1,
  "confidence": float between 0 and 1
}}

Rules:
- Empty category -> empty list, never omit a key.
- Do not fabricate names, dates, or numbers not present or clearly implied.
- Keep each list item a short phrase, not a sentence.
- "importance_score" = how significant this note is for long-term memory.
- "confidence" = how certain you are about this extraction overall."""


def merge_unique(llm_list: List[str], spacy_list: List[str]) -> List[str]:
    """Combine two lists of strings, skipping duplicates (case-insensitive)."""
    seen = {item.lower() for item in llm_list}
    merged = list(llm_list)
    for item in spacy_list:
        if item.lower() not in seen:
            merged.append(item)
            seen.add(item.lower())
    return merged


def extract_memory(processed_note: Dict[str, Any], llm_client) -> Dict[str, Any]:
    """Turn a processed note into a structured memory object."""
    text = processed_note.get("clean_text", "").strip()
    note_id = processed_note.get("note_id") or f"note_{uuid.uuid4().hex[:12]}"
    timestamp = processed_note.get("timestamp") or datetime.now(timezone.utc).isoformat()

    scaffold = run_structural_pass(text)

    prompt = build_prompt(text, scaffold)
    response = call_llm(
    "You are an expert memory extraction system.",
    prompt
)

    cleaned = re.sub(r"^```(json)?|```$", "", response.strip(), flags=re.MULTILINE).strip()
    extracted = json.loads(cleaned)


    entities = extracted.get("entities", {})
    spacy_people = [e["text"] for e in scaffold["entities"] if e["label"] == "PERSON"]
    spacy_orgs = [e["text"] for e in scaffold["entities"] if e["label"] == "ORG"]
    spacy_locs = [e["text"] for e in scaffold["entities"] if e["label"] in ("GPE", "LOC")]
    spacy_dates = [e["text"] for e in scaffold["entities"] if e["label"] == "DATE"]

    return {
        "note_id": note_id,
        "timestamp": timestamp,
        **{field: extracted.get(field, []) for field in MEMORY_FIELDS},
        "entities": {
            "people": merge_unique(entities.get("people", []), spacy_people),
            "organizations": merge_unique(entities.get("organizations", []), spacy_orgs),
            "locations": merge_unique(entities.get("locations", []), spacy_locs),
            "dates": merge_unique(entities.get("dates", []), spacy_dates),
            "other": entities.get("other", []),
        },
        "relationships": extracted.get("relationships") or scaffold["relations"],
        "importance_score": float(extracted.get("importance_score", 0.0)),
        "confidence": float(extracted.get("confidence", 0.0)),
        "source_text": text,
        "extraction_method": "spacy_ner+dependency_parsing+llm_reasoning",
    }


# MEMORY STORAGE 


def upgrade_schema(connection):
    
    cursor = connection.cursor()

    existing_memory_cols = {row[1] for row in cursor.execute("PRAGMA table_info(memories)")}
    if "note_date" not in existing_memory_cols:
        cursor.execute("ALTER TABLE memories ADD COLUMN note_date TEXT")
    if "created_at" not in existing_memory_cols:
        cursor.execute("ALTER TABLE memories ADD COLUMN created_at TEXT")

    existing_topic_cols = {row[1] for row in cursor.execute("PRAGMA table_info(topics)")}
    if "created_at" not in existing_topic_cols:
        cursor.execute("ALTER TABLE topics ADD COLUMN created_at TEXT")

    existing_obs_cols = {row[1] for row in cursor.execute("PRAGMA table_info(observations)")}
    if "created_at" not in existing_obs_cols:
        cursor.execute("ALTER TABLE observations ADD COLUMN created_at TEXT")

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS metadata (
            metadata_id INTEGER PRIMARY KEY AUTOINCREMENT,
            note_id INTEGER,
            key TEXT,
            value TEXT,
            FOREIGN KEY(note_id) REFERENCES notes(note_id)
        )
    """)

    connection.commit()
    print("Schema upgraded (note_date/created_at columns + metadata table).")




def clean_str(value: Any) -> str:
    """Strip a value to a clean string. 'Python ' / ' python' both become 'Python'/'python'."""
    return str(value).strip()


def value_and_confidence(item: Any, fallback_confidence: float) -> Tuple[str, float]:
    """
    Each extracted item can be either:
      - a plain string:        "Learn PyTorch"
      - or a scored dict:      {"value": "Learn PyTorch", "confidence": 0.91}
    Falls back to the note-level confidence when the item has none of its own.
    """
    if isinstance(item, dict):
        value = item.get("value") or item.get("text") or item.get("name") or ""
        confidence = item.get("confidence", fallback_confidence)
    else:
        value = item
        confidence = fallback_confidence
    return clean_str(value), float(confidence)


def topic_name_and_score(item: Any) -> Tuple[str, float]:
    """
    Each topic can be either:
      - a plain string:   "Machine Learning"
      - or a scored dict: {"name": "Machine Learning", "score": 0.93}
    Defaults to a relevance score of 1.0 when the LLM doesn't provide one
    (NOT importance_score — that's a different concept: how important the
    whole note is, not how relevant this particular topic is).
    """
    if isinstance(item, dict):
        name = item.get("name") or item.get("topic") or item.get("text") or ""
        score = item.get("score", 1.0)
    else:
        name = item
        score = 1.0
    return clean_str(name), float(score)



def save_memory_data(connection, processed_note: Dict[str, Any], memory: Dict[str, Any]) -> int:
    """
    Persist a processed note + its extracted memory object into the database.
    Returns the note_id. If the note already exists (same content hash),
    its existing note_id is reused and any sub-tables are still filled in,
    so a previous partial/crashed run doesn't get permanently skipped.
    """

    cursor = connection.cursor()
    now = datetime.now().isoformat()
    note_date = processed_note.get("note_date")


    cursor.execute("SELECT note_id FROM notes WHERE note_hash = ?", (processed_note.get("note_hash"),))
    row = cursor.fetchone()

    if row:
        note_id = row[0]
        print(f"Note already exists (note_id={note_id}) — filling in any missing related data.")
    else:
        cursor.execute(
            """
            INSERT INTO notes (note_hash, note_date, original_text, clean_text, summary, importance, created_at)
            VALUES (?, ?, ?, ?, ?, ?, ?)
            """,
            (
                processed_note.get("note_hash"),
                note_date,
                processed_note.get("original_text"),
                processed_note.get("clean_text"),
                processed_note.get("summary"),
                processed_note.get("importance", memory.get("importance_score", 0.0)),
                now,
            ),
        )
        note_id = cursor.lastrowid
        print(f"Inserted new note (note_id={note_id}).")

    memory_rows = []
    for field in MEMORY_FIELDS:
        for item in memory.get(field, []):
            value, confidence = value_and_confidence(item, memory.get("confidence", 0.0))
            if value:
                memory_rows.append((note_id, field, value, confidence, note_date, now))

    if memory_rows:
        cursor.executemany(
            """
            INSERT INTO memories (note_id, memory_type, memory_value, confidence, note_date, created_at)
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            memory_rows,
        )

    topic_rows = []
    for item in memory.get("topics", []):
        name, score = topic_name_and_score(item)
        if name:
            topic_rows.append((note_id, name, score, now))

    if topic_rows:
        cursor.executemany(
            "INSERT INTO topics (note_id, topic, score, created_at) VALUES (?, ?, ?, ?)",
            topic_rows,
        )

    entity_type_map = {
        "people": "PERSON",
        "organizations": "ORG",
        "locations": "LOC",
        "dates": "DATE",
        "other": "OTHER",
    }
    entity_rows = []
    entities = memory.get("entities", {})
    for key, entity_type in entity_type_map.items():
        for entity_text in entities.get(key, []):
            text = clean_str(entity_text)
            if text:
                entity_rows.append((note_id, text, entity_type))

    if entity_rows:
        cursor.executemany(
            "INSERT INTO entities (note_id, entity, entity_type) VALUES (?, ?, ?)",
            entity_rows,
        )

    relationship_rows = []
    for rel in memory.get("relationships", []):
        subject = clean_str(rel.get("subject", ""))
        predicate = clean_str(rel.get("predicate", ""))
        obj_raw = rel.get("object")
        obj = clean_str(obj_raw) if obj_raw else None
        if subject and predicate:
            relationship_rows.append((note_id, subject, predicate, obj))

    if relationship_rows:
        cursor.executemany(
            "INSERT INTO relationships (note_id, source, relation, target) VALUES (?, ?, ?, ?)",
            relationship_rows,
        )

    embedding = processed_note.get("embedding")
    if embedding is not None:
        embedding_blob = np.asarray(embedding, dtype="float32").tobytes()
        cursor.execute(
            "INSERT OR REPLACE INTO embeddings (note_id, embedding) VALUES (?, ?)",
            (note_id, embedding_blob),
        )

    observation_rows = [
        (note_id, clean_str(obs), now)
        for obs in memory.get("observations", []) if clean_str(obs)
    ]
    if observation_rows:
        cursor.executemany(
            "INSERT INTO observations (note_id, observation, created_at) VALUES (?, ?, ?)",
            observation_rows,
        )

    metadata_rows = [
        (note_id, clean_str(key), clean_str(value))
        for key, value in processed_note.get("metadata", {}).items()
    ]
    if metadata_rows:
        cursor.executemany(
            "INSERT INTO metadata (note_id, key, value) VALUES (?, ?, ?)",
            metadata_rows,
        )

    connection.commit()

    print(
        f"Stored note_id={note_id}: "
        f"{len(memory_rows)} memories, {len(topic_rows)} topics, "
        f"{len(entity_rows)} entities, {len(relationship_rows)} relationships, "
        f"{len(observation_rows)} observations, {len(metadata_rows)} metadata fields."
    )

    return note_id


# MEMORY CONSTRUCTION

def build_vector_index(connection):

    cursor = connection.cursor()

    cursor.execute("""

        SELECT note_id, embedding

        FROM embeddings

        ORDER BY note_id

    """)

    rows = cursor.fetchall()

    index = faiss.IndexFlatIP(EMBEDDING_DIMENSION)

    note_ids = []

    embeddings = []

    for note_id, blob in rows:

        vector = np.frombuffer(

            blob,

            dtype=np.float32

        )

        embeddings.append(vector)

        note_ids.append(note_id)

    if embeddings:

        matrix = np.vstack(embeddings)

        faiss.normalize_L2(matrix)

        index.add(matrix)

    print(f"Vector index contains {len(note_ids)} notes.")

    return index, note_ids

def build_knowledge_graph(connection):

    cursor = connection.cursor()

    graph = nx.MultiDiGraph()

    cursor.execute("""

        SELECT source,

               relation,

               target

        FROM relationships

    """)

    for source, relation, target in cursor.fetchall():

        graph.add_edge(

            source,

            target,

            relation=relation

        )

    print(

        f"Knowledge graph: "

        f"{graph.number_of_nodes()} nodes, "

        f"{graph.number_of_edges()} edges"

    )

    return graph

def build_timeline(connection):

    cursor = connection.cursor()

    cursor.execute("""

        SELECT

            note_date,

            note_id,

            summary

        FROM notes

        ORDER BY note_date

    """)

    timeline = []

    for row in cursor.fetchall():

        timeline.append({

            "date": row[0],

            "note_id": row[1],

            "summary": row[2]

        })

    print(

        f"Timeline contains {len(timeline)} events."

    )

    return timeline

def build_memory_cache(connection):

    cursor = connection.cursor()

    cursor.execute("""

        SELECT

            note_id,

            clean_text,

            summary,

            importance

        FROM notes

    """)

    cache = {}

    for row in cursor.fetchall():

        cache[row[0]] = {

            "text": row[1],

            "summary": row[2],

            "importance": row[3]

        }

    print(

        f"Cache contains {len(cache)} notes."

    )

    return cache

def build_concept_graph(connection):

    cursor = connection.cursor()

    graph = nx.Graph()

    cursor.execute("""

        SELECT

            note_id,

            topic

        FROM topics

    """)

    note_topics = defaultdict(list)

    for note_id, topic in cursor.fetchall():

        note_topics[note_id].append(topic)

    for topics in note_topics.values():

        for i in range(len(topics)):

            for j in range(i + 1, len(topics)):

                graph.add_edge(

                    topics[i],

                    topics[j]

                )

    print(

        f"Concept graph has "

        f"{graph.number_of_nodes()} concepts."

    )

    return graph

def build_entity_graph(connection):

    cursor = connection.cursor()

    graph = nx.Graph()

    cursor.execute("""

        SELECT

            note_id,

            entity

        FROM entities

    """)

    note_entities = defaultdict(list)

    for note_id, entity in cursor.fetchall():

        note_entities[note_id].append(entity)

    for entities in note_entities.values():

        for i in range(len(entities)):

            for j in range(i + 1, len(entities)):

                graph.add_edge(

                    entities[i],

                    entities[j]

                )

    print(

        f"Entity graph has "

        f"{graph.number_of_nodes()} entities."

    )

    return graph



def build_memory_system(connection):

    print("\nBuilding memory system...")

    vector_index, vector_note_ids = build_vector_index(connection)

    knowledge_graph = build_knowledge_graph(connection)

    timeline = build_timeline(connection)

    memory_cache = build_memory_cache(connection)

    concept_graph = build_concept_graph(connection)

    entity_graph = build_entity_graph(connection)

    print("Memory system built.")

    return {

        "vector_index": vector_index,

        "vector_note_ids": vector_note_ids,

        "knowledge_graph": knowledge_graph,

        "timeline": timeline,

        "memory_cache": memory_cache,

        "concept_graph": concept_graph,

        "entity_graph": entity_graph

    }

